# Experiment 2b — Taxonomy Classifier, Tuned Recipe
Fixes for the majority-collapse in 2a: **class-weighted loss**, higher LR (3e-5), warmup, 15 epochs with per-epoch macro-F1 so learning (or collapse) is visible live, and best-checkpoint selection on validation macro-F1.

Also prints the **always-MAN baseline** so Table 2 has an explicit floor.

Runtime → T4 GPU → Run all (~15 min)

In [ ]:
# Cell 1 — GPU check
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — Install (~2 min)
!pip -q install transformers==4.44.2 datasets accelerate scikit-learn

In [ ]:
# Cell 3 — Load HM-800 from HuggingFace
from datasets import load_dataset
import pandas as pd, numpy as np

raw = load_dataset("tanmayvasvani/hindi-misinfo-taxonomy-800",
                   data_files="hindi_misinfo_800_labeled.csv")["train"]
df = raw.to_pandas()
CLASSES = ["FAB","MAN","OLD"]
df = df[df["Category"].isin(CLASSES)].copy()
df["label"] = df["Category"].map({c:i for i,c in enumerate(CLASSES)})
train = df[df["split"]=="train"]; val = df[df["split"]=="val"]; test = df[df["split"]=="test"]
print(f"train {len(train)} / val {len(val)} / test {len(test)}")

# The floor every model must beat:
from sklearn.metrics import f1_score
always_man = np.full(len(test), 1)
floor = f1_score(test["label"], always_man, average="macro")
print(f"ALWAYS-MAN BASELINE macro F1 = {floor:.4f}  <- Exp 2a landed exactly here")

In [ ]:
# Cell 4 — Tokenize
from transformers import AutoTokenizer
import datasets as hfds
tok = AutoTokenizer.from_pretrained("google/muril-base-cased")
def to_ds(d):
    ds = hfds.Dataset.from_pandas(d[["Post","label"]].rename(columns={"Post":"text"}), preserve_index=False)
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)
ds_train, ds_val, ds_test = to_ds(train), to_ds(val), to_ds(test)

In [ ]:
# Cell 5 — Class-weighted fine-tune (~10 min). Watch macro_f1 each epoch:
# rising above 0.22 = learning; flat at 0.22 = collapse again (tell Claude either way)
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import numpy as np, torch, torch.nn as nn

weights = compute_class_weight("balanced", classes=np.array([0,1,2]), y=train["label"].values)
print("Class weights [FAB,MAN,OLD]:", np.round(weights,3))
w = torch.tensor(weights, dtype=torch.float32).cuda()

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=w)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained("google/muril-base-cased", num_labels=3)
args = TrainingArguments(
    output_dir="exp2b_out", num_train_epochs=15,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    learning_rate=3e-5, warmup_ratio=0.1, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    logging_strategy="epoch", report_to="none", seed=42,
    save_safetensors=False)  # MuRIL tensors are non-contiguous; classic .bin format required

trainer = WeightedTrainer(model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
# Cell 6 — TEST RESULTS (paper Table 2, attempt 2)
import json, numpy as np
from sklearn.metrics import classification_report, f1_score, confusion_matrix

pred_out = trainer.predict(ds_test)
preds = np.argmax(pred_out.predictions, axis=1)
y_true = test["label"].values

print("\n=== EXPERIMENT 2b — TEST RESULTS ===")
print(f"Macro F1:    {f1_score(y_true, preds, average='macro'):.4f}   (floor to beat: always-MAN = 0.2202)")
print(f"Weighted F1: {f1_score(y_true, preds, average='weighted'):.4f}\n")
print(classification_report(y_true, preds, target_names=["FAB","MAN","OLD"], digits=4))
print("Confusion [rows=true FAB,MAN,OLD]:")
print(confusion_matrix(y_true, preds))

res = {"macro_f1": float(f1_score(y_true, preds, average="macro")),
       "weighted_f1": float(f1_score(y_true, preds, average="weighted")),
       "report": classification_report(y_true, preds, target_names=["FAB","MAN","OLD"], output_dict=True)}
with open("exp2b_results.json","w") as f: json.dump(res, f, indent=2)

errs = test.copy()
errs["predicted"] = [ ["FAB","MAN","OLD"][p] for p in preds ]
errs = errs[errs["Category"] != errs["predicted"]]
errs[["global_id","Post","Category","predicted","Reason"]].to_csv("exp2b_errors.csv", index=False)
print(f"\nSaved: exp2b_results.json + exp2b_errors.csv ({len(errs)} errors) — download BOTH now.")

## Done
Screenshot Cell 6's output; download **exp2b_results.json** and **exp2b_errors.csv** immediately.
If macro F1 stayed pinned at 0.2202 through all 15 epochs, that's the collapse persisting — report it exactly; it changes the paper's story, honestly.